In [7]:
import json
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from groq import Groq
MODEL = "openai/gpt-oss-20b"
groq = Groq(api_key="XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")


In [2]:

# ---------------------------------------------------------------
# Tools
# ---------------------------------------------------------------
def database(name: str):
    """Search the database by name and return the person's info."""
    time.sleep(1)  # simulate a slow DB call so the parallel speed-up is visible
    data = {
        "yousef": {"name": "yousef", "age": 25, "major": "ai", "address": "cairo"},
        "ali": {"name": "ali", "age": 30, "major": "frontend", "address": "cairo"},
        "sama": {"name": "sama", "age": 27, "major": "data analysis", "address": "cairo"},
    }
    if name not in data:
        raise KeyError(f"'{name}' not found in database")
    return data[name]


def format_value(value) -> str:
    """Turn tool results (dict / list / str) into readable text."""
    if isinstance(value, dict):
        return "\n".join(f"{k}: {v}" for k, v in value.items())
    if isinstance(value, list):
        return "\n\n".join(format_value(v) for v in value)
    return str(value)


def write_file(content):
    """Write data to a text file (final.txt)."""
    with open("final.txt", mode="w", encoding="utf-8") as file:
        file.write(format_value(content))
    return "final.txt written successfully"


TOOLS = {"database": database, "write_file": write_file}


In [3]:
# ---------------------------------------------------------------
# Planner  (now returns dependencies so independent steps can run together)
# ---------------------------------------------------------------
PLANNER_PROMPT = """
You are an AI planning system.

Your job is to create an executable plan for the user's request.

Available tools:

1. database(name) -> searches the database by name (string) and returns a
   dictionary with the person's information (name, age, major, address).

2. write_file(content) -> writes the given content to a text file.

Rules:
- Every step has a "depends_on" list with the ids of steps that must finish
  BEFORE it can start. Use [] if the step needs nothing from other steps.
- Steps that do not depend on each other will be executed IN PARALLEL, so do
  NOT add a dependency unless the step really needs the other step's output.
- To use the output of a previous step, put the placeholder "{{step_N}}"
  (N = id of that step) inside the arguments, and add N to "depends_on".
  Example: {"content": "{{step_1}}\\n\\n{{step_2}}"}
- Never invent or guess data that comes from a tool; always use a placeholder.

Return ONLY valid JSON.

Format:

{
    "steps": [
        {
            "id": 1,
            "task": "...",
            "tool": "...",
            "arguments": {},
            "depends_on": []
        }
    ]
}

Do not execute any tool.
Only create the plan.
"""


def parse_json(text: str):
    """Parse JSON even if the model wrapped it in ```json fences."""
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return json.loads(text)


def create_plan(request: str):
    response = groq.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": PLANNER_PROMPT},
            {"role": "user", "content": request},
        ],
    )
    return parse_json(response.choices[0].message.content)


In [4]:
# ---------------------------------------------------------------
# Dependency handling
# ---------------------------------------------------------------
PLACEHOLDER = re.compile(r"\{\{step_(\d+)\}\}")


def placeholders_in(value):
    """All step ids referenced through {{step_N}} inside an argument value."""
    if isinstance(value, str):
        return {int(n) for n in PLACEHOLDER.findall(value)}
    return set()


def get_dependencies(step: dict) -> set:
    """Declared depends_on + anything referenced through placeholders."""
    deps = set(step.get("depends_on", []))
    for value in step["arguments"].values():
        deps |= placeholders_in(value)
    return deps


def build_levels(plan: dict):
    """
    Group steps into levels. Every step in a level only depends on steps from
    earlier levels, so all steps inside one level can run in parallel.
    """
    steps = {s["id"]: s for s in plan["steps"]}
    deps = {sid: get_dependencies(s) for sid, s in steps.items()}

    for sid, d in deps.items():
        unknown = d - steps.keys()
        if unknown:
            raise ValueError(f"Step {sid} depends on unknown step(s): {unknown}")

    levels, done = [], set()
    remaining = set(steps)
    while remaining:
        ready = sorted(sid for sid in remaining if deps[sid] <= done)
        if not ready:
            raise ValueError("Circular dependency found in the plan")
        levels.append([steps[sid] for sid in ready])
        done |= set(ready)
        remaining -= set(ready)
    return levels


def resolve_arguments(arguments: dict, results: dict):
    """Replace {{step_N}} placeholders with the real result of step N."""
    resolved = {}
    for key, value in arguments.items():
        if isinstance(value, str):
            full = re.fullmatch(r"\{\{step_(\d+)\}\}", value.strip())
            if full:
                resolved[key] = results[int(full.group(1))]
            else:
                resolved[key] = PLACEHOLDER.sub(
                    lambda m: format_value(results[int(m.group(1))]), value
                )
        else:
            resolved[key] = value
    return resolved


In [5]:
# ---------------------------------------------------------------
# Parallel executor
# ---------------------------------------------------------------
def run_step(step: dict, results: dict):
    arguments = resolve_arguments(step["arguments"], results)
    print(f"  [START] step {step['id']} | {step['tool']} | {arguments}")
    start = time.time()
    output = TOOLS[step["tool"]](**arguments)
    print(f"  [DONE ] step {step['id']} | {time.time() - start:.2f}s")
    return step["id"], output


def execute_plan_parallel(plan: dict, max_workers: int = 5):
    results = {}
    levels = build_levels(plan)

    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        for i, level in enumerate(levels, start=1):
            ids = [s["id"] for s in level]
            print(f"\n[LEVEL {i}] running steps {ids} in parallel")

            futures = [pool.submit(run_step, step, results) for step in level]
            for future in as_completed(futures):
                step_id, output = future.result()  # re-raises if a tool failed
                results[step_id] = output

    return dict(sorted(results.items()))


In [6]:


# ---------------------------------------------------------------
# Final answer
# ---------------------------------------------------------------
def generate_final_answer(user_input, plan, results):
    prompt = f"""
Answer the user's request using the execution results.

User request:
{user_input}

Plan:
{json.dumps(plan, indent=2)}

Execution results:
{json.dumps(results, indent=2)}

Give a clear final answer. Only state what is actually in the execution results.
"""
    response = groq.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content


# ---------------------------------------------------------------
# Agent
# ---------------------------------------------------------------
def parallel_planning_agent(question: str):
    print("=" * 60)
    print("Build Planning.")
    print(f"QUESTION: {question}")
    print("=" * 60)

    plan = create_plan(question)
    print(f"PLAN: {json.dumps(plan, indent=2)}")

    print("=" * 60)
    print("Execute Plan (parallel)..")
    start = time.time()
    results = execute_plan_parallel(plan)
    print(f"\nTotal execution time: {time.time() - start:.2f}s")
    print(f"RESULTS: {results}")

    print("=" * 60)
    print("Final Answer")
    return generate_final_answer(question, plan, results)


if __name__ == "__main__":
    q = (
        "search in database for 'yousef', 'ali' and 'sama', "
        "then create a text file and write all their information in it"
    )
    print(parallel_planning_agent(q))

Build Planning.
QUESTION: search in database for 'yousef', 'ali' and 'sama', then create a text file and write all their information in it
PLAN: {
  "steps": [
    {
      "id": 1,
      "task": "Search for 'yousef' in database",
      "tool": "database",
      "arguments": {
        "name": "yousef"
      },
      "depends_on": []
    },
    {
      "id": 2,
      "task": "Search for 'ali' in database",
      "tool": "database",
      "arguments": {
        "name": "ali"
      },
      "depends_on": []
    },
    {
      "id": 3,
      "task": "Search for 'sama' in database",
      "tool": "database",
      "arguments": {
        "name": "sama"
      },
      "depends_on": []
    },
    {
      "id": 4,
      "task": "Write all retrieved information to a file",
      "tool": "write_file",
      "arguments": {
        "content": "{{step_1}}\n\n{{step_2}}\n\n{{step_3}}"
      },
      "depends_on": [
        1,
        2,
        3
      ]
    }
  ]
}
Execute Plan (parallel)..

[LEVEL 1